# Irrigation Need Prediction — ML Final Project

**Course:** Introduction to Machine Learning  
**Dataset:** Kaggle Playground Series S6E4  
**Task:** Multi-class classification — predict irrigation need (Low / Medium / High)

For this project we're using a real-world agricultural dataset that has soil properties, weather data, and crop information to predict how much irrigation a field needs. I'll apply the techniques we covered in class: decision trees, logistic regression, random forests, gradient boosting, neural networks, and ensemble methods. We'll also do some feature engineering and hyperparameter tuning to try to improve the results.

### Project Outline
1. Setup & Imports
2. Load the Data
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. Model Building
   - Decision Tree
   - Logistic Regression
   - Random Forest
   - Gradient Boosting (LightGBM, XGBoost, CatBoost)
   - Neural Network (MLP)
6. Feature Engineering
7. Hyperparameter Tuning (GridSearchCV)
8. Ensemble Method
9. Final Submission

---
## 1. Setup & Imports

In [ ]:
!pip install lightgbm xgboost catboost scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# sklearn models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier

# sklearn utilities
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# gradient boosting libraries
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
N_FOLDS = 5
np.random.seed(SEED)

print('All libraries loaded successfully!')

---
## 2. Load the Data

In [ ]:
# Update these paths if needed
DATA_DIR = 'TrainTestData/'  # folder containing train.csv, test.csv, sample_submission.csv

train = pd.read_csv(DATA_DIR + 'train.csv')
test  = pd.read_csv(DATA_DIR + 'test.csv')
sub   = pd.read_csv(DATA_DIR + 'sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Sample submission shape: {sub.shape}')

In [ ]:
train.head()

In [ ]:
train.dtypes

---
## 3. Exploratory Data Analysis (EDA)

Before building any models we check for missing values, look at the target class distribution, and see how features relate to the target.

In [ ]:
# --- Missing Values ---
missing = train.isnull().sum()
print('Missing values in train:')
print(missing[missing > 0] if missing.any() else 'None!')

In [ ]:
# --- Target Distribution ---
target_counts = train['Irrigation_Need'].value_counts()
print('Target class distribution:')
print(target_counts)
print(f'\nClass balance (%):')
print((target_counts / len(train) * 100).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', color=['steelblue', 'coral', 'seagreen'], ax=ax)
ax.set_title('Target Class Distribution (Irrigation_Need)', fontsize=13)
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Identify feature types ---
TARGET = 'Irrigation_Need'
ID_COL = 'id'

cat_cols = [c for c in train.columns if not pd.api.types.is_numeric_dtype(train[c]) and c != TARGET]
num_cols = [c for c in train.columns if pd.api.types.is_numeric_dtype(train[c]) and c not in [ID_COL, TARGET]]

print(f'Categorical features ({len(cat_cols)}): {cat_cols}')
print(f'Numeric features    ({len(num_cols)}): {num_cols}')


In [ ]:
# --- Categorical feature cardinality ---
print('Unique values per categorical feature:')
for col in cat_cols:
    print(f'  {col}: {train[col].nunique()} unique -> {train[col].unique()[:8]}')

In [ ]:
# --- Numeric feature distributions ---
n_cols_plot = 4
n_rows_plot = (len(num_cols) + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(18, 4 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(train[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap (numeric features) ---
corr = train[num_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Correlation Matrix — Numeric Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature vs Target (boxplots for top numeric features) ---
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols[:8]):
    train.boxplot(column=col, by=TARGET, ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Numeric Features by Irrigation_Need Class', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Preprocessing

I need to encode the categorical features as numbers before feeding them into sklearn models, and also encode the target variable.

In [ ]:
# --- Encode target ---
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_label_map = {v: k for k, v in label_map.items()}

y = train[TARGET].map(label_map).values
print('Target encoded. Class mapping:', label_map)
print('Unique values in y:', np.unique(y))

In [ ]:
# --- Combine train/test for consistent encoding ---
all_data = pd.concat([train.drop(columns=[TARGET]), test], axis=0, ignore_index=True)

# Label-encode categorical columns
le = LabelEncoder()
for col in cat_cols:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

# Split back
n_train = len(train)
train_enc = all_data.iloc[:n_train].copy()
test_enc  = all_data.iloc[n_train:].copy()

feature_cols = [c for c in train_enc.columns if c != ID_COL]
X = train_enc[feature_cols].values
X_test = test_enc[feature_cols].values

print(f'X shape: {X.shape}, X_test shape: {X_test.shape}')

---
## 5. Model Building

I'll start with simpler models (Decision Tree, Logistic Regression) and work up to more complex ones. I'm using stratified k-fold cross-validation to evaluate each model. The metric is balanced accuracy since the classes might not be perfectly balanced.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# helper to evaluate any model with cross-validation
def evaluate_model(model, X, y, cv=skf, model_name='Model'):
    scores = cross_val_score(model, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
    print(f'{model_name}: {scores.mean():.4f} ± {scores.std():.4f}')
    return scores

results = {}  # store scores for comparison at the end

In [ ]:
# --- Decision Tree ---
# Starting with a decision tree since it's easy to interpret and was the first
# tree-based method we covered in class.
print('=== Decision Tree ===')
dt = DecisionTreeClassifier(max_depth=10, random_state=SEED, class_weight='balanced')
dt_scores = evaluate_model(dt, X, y, model_name='Decision Tree')
results['Decision Tree'] = dt_scores.mean()

# visualize a shallow version so it's actually readable
dt_viz = DecisionTreeClassifier(max_depth=3, random_state=SEED, class_weight='balanced')
dt_viz.fit(X, y)

plt.figure(figsize=(20, 6))
plot_tree(dt_viz, feature_names=feature_cols, class_names=['Low', 'Medium', 'High'],
          filled=True, fontsize=8)
plt.title('Decision Tree (max_depth=3, simplified for visualization)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Logistic Regression ---
# Need to scale features first since LR is sensitive to feature magnitudes.
print('=== Logistic Regression ===')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced')
lr_scores = evaluate_model(lr, X_scaled, y, model_name='Logistic Regression')
results['Logistic Regression'] = lr_scores.mean()

# --- Random Forest ---
# Random Forest builds many decision trees and averages their predictions —
# reduces the overfitting we saw with a single decision tree.
print('\n=== Random Forest ===')
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, class_weight='balanced', n_jobs=-1)
rf_scores = evaluate_model(rf, X, y, model_name='Random Forest')
results['Random Forest'] = rf_scores.mean()

---
## 5b. Neural Network (MLP)

We covered neural networks in the Deep Learning unit. I'm using sklearn's MLPClassifier (Multi-Layer Perceptron) with two hidden layers. MLPs need scaled input features, same as logistic regression.

In [ ]:
print('=== Neural Network (MLP) ===')

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=200,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.1,
)

mlp_scores = evaluate_model(mlp, X_scaled, y, model_name='Neural Network (MLP)')
results['Neural Network (MLP)'] = mlp_scores.mean()

---
## 5c. Gradient Boosting Models

These are boosting-based ensemble methods — each tree tries to correct the errors of the previous one. I'm testing three popular libraries: LightGBM, XGBoost, and CatBoost.

In [ ]:
# --- LightGBM ---
print('=== LightGBM ===')

LGBM_PARAMS = dict(
    objective='multiclass',
    num_class=3,
    metric='multi_logloss',
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

lgbm = lgb.LGBMClassifier(**LGBM_PARAMS)
lgbm_scores = evaluate_model(lgbm, X, y, model_name='LightGBM')
results['LightGBM'] = lgbm_scores.mean()

In [ ]:
# --- Feature Importance (LightGBM) ---
lgbm.fit(X, y)

feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgbm.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', palette='viridis')
plt.title('LightGBM Feature Importance', fontsize=13)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(feat_imp.to_string(index=False))

In [ ]:
# --- XGBoost ---
print('=== XGBoost ===')

XGB_PARAMS = dict(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

xgb_model = xgb.XGBClassifier(**XGB_PARAMS)
xgb_scores = evaluate_model(xgb_model, X, y, model_name='XGBoost')
results['XGBoost'] = xgb_scores.mean()

In [ ]:
# --- CatBoost ---
# CatBoost can handle categorical features natively without needing to encode them first,
# which is a nice feature for this dataset.
print('=== CatBoost ===')

CBC_PARAMS = dict(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    random_seed=SEED,
    verbose=0,
)

cbc = CatBoostClassifier(**CBC_PARAMS)
cbc_scores = evaluate_model(cbc, X, y, model_name='CatBoost')
results['CatBoost'] = cbc_scores.mean()

In [ ]:
# --- Model Comparison So Far ---
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Balanced Accuracy (CV)': list(results.values())
}).sort_values('Balanced Accuracy (CV)', ascending=False)

print(comparison.to_string(index=False))

plt.figure(figsize=(9, 5))
sns.barplot(data=comparison, x='Balanced Accuracy (CV)', y='Model', palette='viridis')
plt.title('Model Comparison — 5-Fold CV Balanced Accuracy')
plt.xlim(comparison['Balanced Accuracy (CV)'].min() - 0.05, 1.0)
plt.tight_layout()
plt.show()

---
## 6. Feature Engineering

I created some new features based on combinations of existing ones that seem useful for predicting irrigation need (e.g., high temperature + low rainfall probably means more irrigation is needed).

In [ ]:
def engineer_features(df):
    df = df.copy()

    # rainfall × humidity — high values of both suggest low irrigation need
    df['Rain_x_Humidity'] = df['Rainfall_mm'] * df['Humidity']

    # temperature ÷ rainfall — higher ratio = drier conditions = more irrigation
    df['Temp_to_Rain'] = df['Temperature_C'] / (df['Rainfall_mm'] + 1)

    # soil moisture relative to previous irrigation
    df['Moisture_prev_ratio'] = df['Soil_Moisture'] / (df['Previous_Irrigation_mm'] + 1)

    # rough water deficit proxy
    df['Water_deficit'] = df['Temperature_C'] * df['Sunlight_Hours'] - df['Rainfall_mm'] - df['Soil_Moisture']

    # total water in field (area × moisture)
    df['Total_soil_moisture'] = df['Field_Area_hectare'] * df['Soil_Moisture']

    # wind drying effect
    df['Wind_x_Temp'] = df['Wind_Speed_kmh'] * df['Temperature_C']

    return df


train_fe = engineer_features(train.drop(columns=[TARGET]))
test_fe  = engineer_features(test)

all_data_fe = pd.concat([train_fe, test_fe], axis=0, ignore_index=True)
for col in cat_cols:
    all_data_fe[col] = LabelEncoder().fit_transform(all_data_fe[col].astype(str))

train_fe_enc = all_data_fe.iloc[:n_train].copy()
test_fe_enc  = all_data_fe.iloc[n_train:].copy()

feature_cols_fe = [c for c in train_fe_enc.columns if c != ID_COL]
X_fe      = train_fe_enc[feature_cols_fe].values
X_test_fe = test_fe_enc[feature_cols_fe].values

scaler_fe = StandardScaler()
X_fe_scaled      = scaler_fe.fit_transform(X_fe)
X_test_fe_scaled = scaler_fe.transform(X_test_fe)

print(f'Features after engineering: {len(feature_cols_fe)} (was {len(feature_cols)})')
print('New features:', [c for c in feature_cols_fe if c not in feature_cols])

In [ ]:
# Re-evaluate LightGBM with the new features to see if engineering helped
print('=== LightGBM with Feature Engineering ===')
lgbm_fe = lgb.LGBMClassifier(**LGBM_PARAMS)
lgbm_fe_scores = evaluate_model(lgbm_fe, X_fe, y, model_name='LightGBM + FE')
results['LightGBM + FE'] = lgbm_fe_scores.mean()

print(f'\nImprovement: {lgbm_fe_scores.mean() - lgbm_scores.mean():+.4f}')

---
## 7. Hyperparameter Tuning (GridSearchCV)

We'll use GridSearchCV to tune the Random Forest hyperparameters. It tries every combination of parameters in the grid and picks the best one using cross-validation.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
}

# sample 20% of data to keep tuning fast
sample_idx = np.random.choice(len(X_fe), size=int(0.2 * len(X_fe)), replace=False)

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=SEED, class_weight='balanced', n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1,
)

rf_grid.fit(X_fe[sample_idx], y[sample_idx])

print(f'Best parameters: {rf_grid.best_params_}')
print(f'Best CV balanced accuracy: {rf_grid.best_score_:.4f}')

In [ ]:
# evaluate the tuned RF on 5-fold CV
print('=== Tuned Random Forest ===')
rf_tuned = rf_grid.best_estimator_
rf_tuned_scores = evaluate_model(rf_tuned, X_fe, y, model_name='Tuned Random Forest')
results['Tuned Random Forest'] = rf_tuned_scores.mean()

---
## 8. Ensemble Method (Voting Classifier)

We learned about ensemble methods in class, combining multiple models usually gives better results than any single model. We're using a soft voting classifier that averages the predicted probabilities from each model. We left Random Forest out of the final ensemble since LightGBM already covers the tree-based side and scored higher on its own.



In [ ]:
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=500, random_state=SEED, class_weight='balanced'))
])

ensemble = VotingClassifier(
    estimators=[
        ('lr', lr_pipe),
        ('lgbm', lgb.LGBMClassifier(**LGBM_PARAMS)),
    ],
    voting='soft',
    n_jobs=-1,
)

print('=== Voting Ensemble (LR + LightGBM) ===')
ensemble_scores = evaluate_model(ensemble, X_fe, y, model_name='Voting Ensemble')
results['Voting Ensemble'] = ensemble_scores.mean()

In [ ]:
# --- Final Model Comparison ---
final_comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Balanced Accuracy (CV)': list(results.values())
}).sort_values('Balanced Accuracy (CV)', ascending=False)

print(final_comparison.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=final_comparison, x='Balanced Accuracy (CV)', y='Model', palette='viridis')
plt.title('Final Model Comparison — 5-Fold CV Balanced Accuracy')
plt.xlim(final_comparison['Balanced Accuracy (CV)'].min() - 0.05, 1.0)
plt.tight_layout()
plt.show()

---
## 9. Final Submission

Train the best model (ensemble) on the full training set and generate predictions for the test set.

In [ ]:
# Train the final ensemble on all training data
print('Training final ensemble on full training data...')
ensemble.fit(X_fe, y)

final_pred_classes = ensemble.predict(X_test_fe)
inv_label_map = {0: 'Low', 1: 'Medium', 2: 'High'}
final_pred_labels = [inv_label_map[c] for c in final_pred_classes]

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_pred_labels
})

print('Submission class distribution:')
print(submission['Irrigation_Need'].value_counts())
print()
print(submission.head(10))

In [ ]:
SUBMISSION_PATH = 'submission.csv'
submission.to_csv(SUBMISSION_PATH, index=False)
print(f'Submission saved to: {SUBMISSION_PATH}')

In [ ]:
# Verify format matches sample submission
print('Sample submission format:')
print(sub.head())
print()
print('Our submission format:')
print(submission.head())
print()
print('Shapes match:', submission.shape == sub.shape)
print('Columns match:', list(submission.columns) == list(sub.columns))
print('Valid labels:', set(submission['Irrigation_Need']).issubset({'Low', 'Medium', 'High'}))